# Teletransporte quantico

Resumo direto - A ideia é transferir um estado quântico desconhecido de Alice para Bob usando apenas um par de Bell compartilhado e 2 bit clássicos

O teletransporte usa 3 recursos juntos - o par de Bell pré-compartilhado como "canal quântico", a medição de Bell de Alice que projeta o sistema, e os 2 bit clássicos que carregam qual das 4 correções de Pauli Bob precisa aplicar. Retire qualquer um dos três e o protocolo falha. É também a demnonstração mais clara do no-cloning theorem na prática: q0 é destruído no momento da medição

#### Construção do circuito

In [2]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# ── 1. Definir o estado |ψ⟩ que Alice quer teletransportar ──────────────
# Qualquer estado de 1 qubit: α|0⟩ + β|1⟩
# Vamos usar |+⟩ = (|0⟩ + |1⟩)/√2 como exemplo
theta = np.pi / 2   # mude para testar outros estados
phi   = 0.0

# Estado alvo para verificação posterior
psi = Statevector([np.cos(theta/2),
                   np.exp(1j * phi) * np.sin(theta/2)])
print(f"Estado a teletransportar: {psi}")

# ── 2. Construir o circuito ─────────────────────────────────────────────
qr = QuantumRegister(3, 'q')  # q0=mensagem, q1=Alice, q2=Bob
cr = ClassicalRegister(2, 'c')
qc = QuantumCircuit(qr, cr)

# Preparar |ψ⟩ em q0
qc.h(0)           # cria |+⟩ — substitua por qualquer prep que quiser

qc.barrier()

# Criar par de Bell entre q1 (Alice) e q2 (Bob)
qc.h(1)
qc.cx(1, 2)       # q1 e q2 agora compartilham |Φ⁺⟩

qc.barrier()

# Operação de Bell de Alice em q0 e q1
qc.cx(0, 1)
qc.h(0)

qc.barrier()

# Medir q0 e q1 → bits clássicos m0 e m1
qc.measure(0, 0)  # m0
qc.measure(1, 1)  # m1

qc.barrier()

# Correções de Bob em q2 baseadas nos bits clássicos
# "se m1=1 → aplica X"  /  "se m0=1 → aplica Z"
with qc.if_test((cr[1], 1)): # X condicional
    qc.x(2)
with qc.if_test((cr[0], 1)): # Z condicional
    qc.z(2)

print(qc.draw('text'))

Estado a teletransportar: Statevector([0.70710678+0.j, 0.70710678+0.j],
            dims=(2,))
     ┌───┐ ░            ░      ┌───┐ ░ ┌─┐    ░                           »
q_0: ┤ H ├─░────────────░───■──┤ H ├─░─┤M├────░───────────────────────────»
     └───┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐ ░                           »
q_1: ──────░─┤ H ├──■───░─┤ X ├──────░──╫─┤M├─░───────────────────────────»
           ░ └───┘┌─┴─┐ ░ └───┘      ░  ║ └╥┘ ░   ┌──────  ┌───┐ ───────┐ »
q_2: ──────░──────┤ X ├─░────────────░──╫──╫──░───┤ If-0  ─┤ X ├  End-0 ├─»
           ░      └───┘ ░            ░  ║  ║  ░   └──╥───  └───┘ ───────┘ »
                                        ║  ║    ┌────╨────┐               »
c: 2/═══════════════════════════════════╩══╩════╡ c_1=0x1 ╞═══════════════»
                                        0  1    └─────────┘               »
«                               
«q_0: ──────────────────────────
«                               
«q_1: ──────────────────────────
«       ┌────

#### Simulação

In [3]:
# ── 4. Simulação realista com medições ─────────────────────────────────
sim = AerSimulator()
job = sim.run(qc, shots=1024)
counts = job.result().get_counts()
print(f"\nContagens (m0 m1): {counts}")
# Espera: distribuição uniforme entre 00, 01, 10, 11
# O estado de q2 estará correto em TODOS os casos graças às correções


Contagens (m0 m1): {'11': 245, '01': 262, '00': 259, '10': 258}


#### Exemplo mais sofisticado

In [5]:
# ── 5. Teste com vários estados ─────────────────────────────────────────
def teleport(theta, phi=0.0):
    qc = QuantumCircuit(3, 2)

    # Preparar estado alvo em q0
    qc.ry(theta, 0)
    qc.rz(phi, 0)

    # Par de Bell
    qc.h(1)
    qc.cx(1, 2)

    # Protocolo Alice
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    # Correções Bob
    with qc.if_test((qc.cregs[0][1], 1)): # X condicional
        qc.x(2)
    with qc.if_test((qc.cregs[0][0], 1)): # Z condicional
        qc.z(2)

    return qc

# Testar nos 6 estados dos eixos da esfera de Bloch
estados = {
    "|0⟩":  (0,        0),
    "|1⟩":  (np.pi,    0),
    "|+⟩":  (np.pi/2,  0),
    "|-⟩":  (np.pi/2,  np.pi),
    "|i⟩":  (np.pi/2,  np.pi/2),
    "|-i⟩": (np.pi/2, -np.pi/2),
}

for nome, (t, p) in estados.items():
    qc = teleport(t, p)
    # (verificação de fidelidade aqui)
    print(f"Teletransportando {nome}: theta={t:.3f}, phi={p:.3f} ✓")

Teletransportando |0⟩: theta=0.000, phi=0.000 ✓
Teletransportando |1⟩: theta=3.142, phi=0.000 ✓
Teletransportando |+⟩: theta=1.571, phi=0.000 ✓
Teletransportando |-⟩: theta=1.571, phi=3.142 ✓
Teletransportando |i⟩: theta=1.571, phi=1.571 ✓
Teletransportando |-i⟩: theta=1.571, phi=-1.571 ✓
